## Tea Leaf Reliability Research

Start with **demo** to check that everything runs. After downloading the [SLTeaLeaf v5 images](https://data.mendeley.com/datasets/sjmy6k24d6/5), switch to **public** to train your first tea model. Use **field** only with newly collected, expert-confirmed photos and genuine estate/plant IDs.

This notebook runs the project scripts; it does not bundle real images. Synthetic scores are software checks only. Use a Python 3.11/3.12 runtime. A GPU can speed up the real runs.

In [ ]:
from pathlib import Path
import shutil
import stat
import subprocess
import sys
import zipfile

workspace = Path("/content/tea_research_workspace")
project = workspace / "tea_leaf_reliability"
if not (project / "tea_research" / "__main__.py").is_file():
    from google.colab import files
    incoming = files.upload()  # Select Tea_Leaf_Research_Project.zip
    zip_names = [name for name in incoming if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one project ZIP.")
    workspace.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_names[0]) as z:
        for item in z.infolist():
            dest = (workspace / item.filename).resolve()
            if (not dest.is_relative_to(project.resolve())
                or stat.S_ISLNK(item.external_attr >> 16)):
                raise ValueError("Unexpected path or link in the ZIP.")
        z.extractall(workspace)
if not (project / "tea_research" / "__main__.py").is_file():
    raise FileNotFoundError("The ZIP does not contain the project package.")
print(project)

In [ ]:
if sys.version_info[:2] not in {(3, 11), (3, 12)}:
    raise RuntimeError("Use a Python 3.11 or 3.12 environment for these pinned dependencies.")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project / "requirements.txt")],
               check=True)

def cli(*args):
    subprocess.run([sys.executable, "-m", "tea_research", *map(str, args)],
                   cwd=project, check=True)

cli("audit-public-metadata")

### Set the mode and paths

Leave the defaults for the first run. For real data, extract the image archive before setting IMAGE_ROOT. To use folders in Google Drive, set USE_DRIVE = True and point IMAGE_ROOT/ANNOTATIONS to the mounted folders. Keep a new RUN_TAG for each fresh run. For the field mode, choose a held-out estate and the classes agreed during your pilot.

In [ ]:
MODE = "demo"                        # demo, public, or field
RUN_TAG = "first_demo"
MODEL = "mobilenetv3small"             # or efficientnetb0
SEED = 11
USE_DRIVE = False
IMAGE_ROOT = Path("/content/SLTeaLeaf_v5")
ANNOTATIONS = Path("/content/field_annotations.csv")
HOLDOUT = "E01"
FIELD_CLASSES = "healthy,gray_blight,blister_blight"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if MODE not in {"demo", "public", "field"}:
    raise ValueError("Choose demo, public or field.")
if not RUN_TAG or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in RUN_TAG):
    raise ValueError("Use letters, digits, underscores or hyphens for RUN_TAG.")

In [ ]:
work = project / "work" / RUN_TAG
run = project / "runs" / MODE / RUN_TAG

if MODE == "demo":
    cli("make-demo", "--output", work)
    img_root = work / "images"
    manifest = work / "splits" / "fold_S01.csv"
    training_options = [
        "--weights", "none", "--image-size", 64, "--batch-size", 8,
        "--head-epochs", 1, "--finetune-epochs", 1, "--representative", 18,
        "--bootstrap", 50,
    ]
else:
    img_root = IMAGE_ROOT
    if MODE == "public":
        cli("prepare-public", "--image-root", img_root, "--output", work)
        manifest = work / "public.csv"
    else:
        cli("prepare-field", "--annotations", ANNOTATIONS, "--image-root", img_root,
            "--classes", FIELD_CLASSES, "--holdout", HOLDOUT, "--output", work)
        manifest = work / f"fold_{HOLDOUT}.csv"
    training_options = ["--weights", "imagenet", "--batch-size", 32, "--bootstrap", 2000]

cli("run", "--manifest", manifest, "--image-root", img_root, "--output", run,
    "--model", MODEL, "--seed", SEED, *training_options)

### Inspect one prediction and download your run

The report records all four FP32/INT8 and raw/calibrated conditions. Demo outputs contain no evidence about real disease recognition. Use the CLI's separate train, export and calibrate commands for development before opening a final test result.

In [ ]:
import csv
import json
with (run / "manifest.csv").open(encoding="utf-8", newline="") as f:
    example = next(row for row in csv.DictReader(f) if row["split"] == "test")
cli("predict", "--run", run, "--image", img_root / example["image_path"])

report = json.loads((run / "evaluation.json").read_text(encoding="utf-8"))
print("Protocol:", report["protocol"])
print("Files:", ", ".join(sorted(p.name for p in run.iterdir() if p.is_file())))
print("See evaluation.json for per-class results and paired intervals.")

In [ ]:
# Optional: download the trained checkpoint, exports and reports.
from google.colab import files
archive = shutil.make_archive(str(workspace / f"{RUN_TAG}_outputs"), "zip", root_dir=run)
files.download(archive)